# Wine Peer
## Advanced Machine Learning — Final Project

---

### Product Description

Wine Peer is a wine recommendation app. Photograph your food — that is the only input. The app returns three wine recommendations — one that complements the food's flavor, one that contrasts it, and one that balances it — each with a real tasting note from a Vivino user and an approval percentage.

The pipeline is three independent layers:

```
PHOTO  →  CNN  →  food label  →  food flavor profile  →  Word2Vec similarity  →  grape varieties (Complement / Contrast / Balance)  →  BiLSTM review retrieval  →  flavor language + rating %
```

---

### Example Output

| | |
|---|---|
| **Input** | Photo of pizza margherita |
| **CNN output** | Pizza (94% confidence) |
| **Food flavor profile** | savory · salty · cheesy · rich · fatty · tomato |


| Pairing | Grape | Wine | Drinker Quote | Vivino |
|---|---|---|---|---|
| **Complement** — amplifies the food's flavors | Sangiovese | Chianti Classico Riserva 2019 | *"Deep cherry and dried herbs — wraps around the tomato sauce like it was made for it."* | 92% users |
| **Contrast** — cuts through and refreshes | Chardonnay | Chablis Premier Cru 2021 | *"Bone-dry mineral acidity cuts straight through the richness. Resets every bite."* | 89% users |
| **Balance** — neutral crowd pleaser | Pinot Grigio | Santa Margherita Alto Adige 2022 | *"Light, clean and gently fruited. Gets out of the way and lets the pizza do the talking."* | 86% users |

---

### Models

| Component | Dataset | Task |
|---|---|---|
| CNN (scratch) | Food-101 (101k images) | 101-class food classification |
| CNN (ResNet-50) | Food-101 | Same — compare vs. scratch |
| LSTM baseline | WineSensed (824k reviews) | 15-class grape variety classification |
| BiLSTM + attention | WineSensed | Same — compare vs. unidirectional; also used for representative review retrieval |
| Word2Vec pairing | Google News (pre-trained) + WineSensed (fine-tuned) | Flavor embedding similarity — food flavor keywords → nearest grape variety vector |
| Joint Model *(+10 pts)* | Food-101 + WineSensed pairs | Food-wine compatibility: 0/1 |

---

### Pairing Logic

Wine recommendations are computed by flavor embedding similarity, not hardcoded rules:

1. **Load pre-trained Word2Vec** (Google News, 300-d) — the model already knows everyday food language: *tomato*, *fatty*, *smoky*, *spicy*. No food word needs to be translated first.
2. **Fine-tune on WineSensed reviews** — adapts the model to wine tasting vocabulary: *Sangiovese*, *tannic*, *cassis*, *terroir* land in the right neighborhood (~10–15 min CPU, gensim `build_vocab` + `train`).
3. **Build grape embeddings** — average all word vectors across reviews for each of the top 15 grape varieties → 15 grape vectors.
4. **Food flavor table** — each Food-101 class has three sets of flavor keywords (one per pairing type). Because Word2Vec starts from Google News, keywords can freely use natural food language (*tomato*, *fatty*, *rich*) as well as wine tasting words.
5. **At inference** — embed the relevant keyword set, compute cosine similarity against 15 grape vectors, return the top match per pairing type. Then retrieve the highest-rated real wine of that grape with the most representative Vivino review.

| Pairing Type | Keyword intent | Example for pizza |
|---|---|---|
| **Complement** | Flavors that echo and amplify the food | `savory tomato earthy herbaceous rich fatty` |
| **Contrast** | Flavors that cut through and refresh | `mineral crisp citrus steely dry acidic` |
| **Balance** | Approachable, crowd-safe | `light fruity clean gentle soft easy` |

---

### Why Grape Classification (not Wine Types)

Classifying by grape variety rather than broad wine type (Red/White/Rosé) makes the task **harder and more meaningful**:

- The model must learn the flavor language specific to each grape — *"cassis and cedar"* for Cabernet Sauvignon vs *"strawberry and forest floor"* for Pinot Noir
- The recommendation output is more specific and useful to the user: *"you'd like a Sangiovese"* rather than *"you'd like a Red"*
- Word2Vec embeddings per grape place varieties in a fine-grained flavor space where, for example, Syrah and Malbec cluster near each other but far from Riesling

**The 15 grape classes** are selected by frequency in WineSensed and cover ~85% of all reviews:

| Reds | Whites | Other |
|---|---|---|
| Cabernet Sauvignon · Merlot · Pinot Noir · Syrah · Malbec · Sangiovese · Tempranillo · Grenache · Zinfandel | Chardonnay · Sauvignon Blanc · Riesling · Pinot Grigio · Viognier · Chenin Blanc | — |

---

### Review Retrieval (BiLSTM)

After identifying the three recommended grape varieties, the BiLSTM encoder finds the single most representative real Vivino review per grape — the review whose hidden-state vector sits closest to the grape centroid. No text is generated; all quotes are genuine.

The Vivino rating for the selected wine is converted to a user approval percentage: `rating / 5.0 × 100`.

---

### Joint Model Design

The joint model learns food-wine compatibility at the feature level:

- **Positive pairs:** (food image, wine review) where Word2Vec pairing scores the pair as compatible → label `1`
- **Negative pairs:** (food image, wine review) for an incompatible grape variety → label `0`
- **Architecture:** frozen CNN encoder (512-d) + frozen BiLSTM encoder (256-d) → concat (768-d) → FC → sigmoid

---

### Datasets

| Dataset | Content | Used for |
|---|---|---|
| Food-101 (`torchvision.datasets.Food101`) | 101,000 labeled food images | CNN training |
| WineSensed (`Dakhoo/L2T-NeurIPS-2023`) | 824k real Vivino reviews, wine name, grape, rating, country, region | BiLSTM training · Word2Vec fine-tuning · review retrieval |
| Food flavor table (embedded, Section 2.4) | 101 foods × 3 keyword sets (complement / contrast / balance) | Word2Vec pairing · joint model labels |

**BiLSTM labels:** primary grape variety → top 15 classes by frequency (covers ~85% of all reviews)

---

### Development Phases

| Phase | Environment | Sections | Needs GPU | What happens |
|---|---|---|---|---|
| **Phase 1** | VS Code (CPU) | 1, 2, 3, 4, 6, 10, 11 | No | Environment setup. Data loading. EDA (images + text). Text preprocessing. Word2Vec fine-tuning + grape embeddings. LSTM baseline. BiLSTM with attention. |
| **Phase 2** | Google Colab (GPU) | 5, 7, 8, 13 | **Yes** | Image preprocessing + data loaders. CNN from scratch. CNN ResNet-50. Joint model full training. |
| **Phase 3** | VS Code (CPU) | 9, 12, 14, 15, deployment | No | Grad-CAM explainability. BiLSTM attention explainability. Recommendation card + 20-example table. Business framing + ethics. HF Spaces deployment. |

---

### Notebook Structure

| Section | Content |
|---|---|
| 1 | Environment setup — dependencies and imports |
| 2 | Data loading — Food-101 + WineSensed + food flavor table |
| 3 | EDA — image dataset (class distribution, sample grid) |
| 4 | EDA — text dataset (review length, grape distribution, word clouds) |
| 5 | Image preprocessing and data loaders |
| 6 | Text preprocessing and data loaders |
| 7 | CNN — custom architecture (trained from scratch) |
| 8 | CNN — ResNet-50 (transfer learning) |
| 9 | CNN explainability — Grad-CAM |
| 10 | LSTM — unidirectional baseline |
| 11 | BiLSTM — bidirectional with attention |
| 12 | BiLSTM explainability — attention weights |
| 13 | Joint model — food-wine compatibility classifier *(+10 bonus)* |
| 14 | Business integration — recommendation card and 20-example table |
| 15 | Business framing, ethics, and team contributions |

---


## Section 1 — Environment Setup

Quick environment check — confirm Colab + CUDA before installing packages.


In [1]:
import sys, torch

IN_COLAB = "google.colab" in sys.modules
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"{'Environment':<20} {'Google Colab' if IN_COLAB else 'Local'}")
print(f"{'Device':<20} {DEVICE}")
print(f"{'CUDA available':<20} {torch.cuda.is_available()}")

if not IN_COLAB:
    print("\n⚠  Not running in Google Colab — GPU sections (5, 7, 8, 13) require Colab.")
elif not torch.cuda.is_available():
    print("\n⚠  CUDA not available — enable the GPU runtime in Colab: Runtime → Change runtime type → T4 GPU.")
else:
    print("\n✓ Environment check passed — Colab + CUDA confirmed.")


Environment          Google Colab
Device               cuda
CUDA available       True

✓ Environment check passed — Colab + CUDA confirmed.


### 1.1 — pip installs

If you need to install all required packages, run the cell below.
It auto-detects the environment (local CPU vs. Google Colab GPU) and installs only what is missing.


In [ ]:
import subprocess, sys, importlib.util

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

def is_installed(import_name):
    """Check if a package is installed without importing it (avoids __init__ side-effects)."""
    return importlib.util.find_spec(import_name) is not None

# ── Detect environment ────────────────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")

# ── Install PyTorch with correct backend ──────────────────────────────────────
if is_installed("torch"):
    import torch
    print(f"  OK  torch {torch.__version__}")
else:
    print("  INSTALLING  torch ...")
    if IN_COLAB:
        pip_install("torch", "torchvision", "torchaudio",
                    "--index-url", "https://download.pytorch.org/whl/cu118")
    else:
        pip_install("torch", "torchvision", "torchaudio",
                    "--index-url", "https://download.pytorch.org/whl/cpu")
    print("  DONE  torch")

# ── Install remaining packages ────────────────────────────────────────────────
# Keys = import name (used for find_spec), values = pip install name
PACKAGES = {
    "datasets":   "datasets",    # HuggingFace datasets — for WineSensed + Food-101
    "nltk":       "nltk",
    "torchcam":   "torchcam",
    "wordcloud":  "wordcloud",
    "matplotlib": "matplotlib",
    "seaborn":    "seaborn",
    "pandas":     "pandas",
    "sklearn":    "scikit-learn",
    "PIL":        "Pillow",
    "ipywidgets": "ipywidgets",   # needed for tqdm notebook progress bars
}

for pkg, install_name in PACKAGES.items():
    if is_installed(pkg):
        print(f"  OK  {pkg}")
    else:
        print(f"  INSTALLING  {pkg} ...")
        pip_install(install_name)
        print(f"  DONE  {pkg}")

print("\nAll packages ready.")


Environment: Google Colab
  OK  torch 2.10.0+cu128
  OK  kaggle
  OK  nltk
  OK  torchcam
  OK  wordcloud
  OK  matplotlib
  OK  seaborn
  OK  pandas
  OK  sklearn
  OK  PIL
  OK  ipywidgets

All packages ready.


### 1.2 — Library imports

Import all libraries, set the random seed, detect the device, and create project folders.


In [3]:

# ── Fix numpy/pandas binary incompatibility ───────────────────────────────────
# Run this cell once, then Runtime → Restart runtime, then re-run from Section 1.
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
                       "numpy", "pandas"])
print("numpy and pandas reinstalled — restart the runtime now (Runtime → Restart runtime).")


numpy and pandas reinstalled — restart the runtime now (Runtime → Restart runtime).


In [4]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import sys
import random
import string
import importlib
from pathlib import Path
from collections import Counter

# Suppress HF Hub symlinks warning on Windows (fallback to copies, still works)
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

# ── Numeric / data ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from wordcloud import WordCloud
from PIL import Image

# ── PyTorch core ──────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ── Computer vision ───────────────────────────────────────────────────────────
import torchvision.transforms as T
import torchvision.models as models
import torchvision.datasets as tv_datasets

# ── NLP ───────────────────────────────────────────────────────────────────────
import nltk
from nltk.tokenize import word_tokenize

# ── Sklearn utilities ─────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score

# ── Environment ───────────────────────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Project directories ───────────────────────────────────────────────────────
BASE_DIR = Path(".")
WEIGHTS  = BASE_DIR / "weights"
FIGURES  = BASE_DIR / "figures"
DEPLOY   = BASE_DIR / "deployment"
DATA_DIR = BASE_DIR / "data"

for d in [WEIGHTS, FIGURES, DEPLOY, DATA_DIR]:
    d.mkdir(exist_ok=True)

# ── NLTK data ─────────────────────────────────────────────────────────────────
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

# ── Library versions ──────────────────────────────────────────────────────────
libs = [
    "torch", "torchvision", "numpy", "pandas",
    "matplotlib", "seaborn", "PIL", "sklearn",
    "nltk", "wordcloud",
]

print(f"{'Library':<20} {'Version'}")
print("-" * 35)
for lib in libs:
    try:
        mod = importlib.import_module(lib)
        print(f"{lib:<20} {getattr(mod, '__version__', 'n/a')}")
    except ImportError:
        print(f"{lib:<20} NOT INSTALLED")

print()
print(f"{'Environment':<20} {'Google Colab' if IN_COLAB else 'Local'}")
print(f"{'Device':<20} {DEVICE}")
print(f"{'CUDA available':<20} {torch.cuda.is_available()}")
print(f"{'Directories':<20} {[str(d) for d in [WEIGHTS, FIGURES, DEPLOY, DATA_DIR]]}")
print("\nSection 1 complete.")


Library              Version
-----------------------------------
torch                2.10.0+cu128
torchvision          0.25.0+cu128
numpy                2.4.4
pandas               3.0.2
matplotlib           3.10.0
seaborn              0.13.2
PIL                  11.3.0
sklearn              1.6.1
nltk                 3.9.1
wordcloud            1.9.6

Environment          Google Colab
Device               cuda
CUDA available       True
Directories          ['weights', 'figures', 'deployment', 'data']

Section 1 complete.


---

## Section 2 — Data Loading

Load all three data sources used in this project. Each subsection loads one source, validates it, and prints a summary. No preprocessing yet — raw data only.

---

### 2.1 — Load Food-101

Load the image dataset via `torchvision.datasets.Food101` (downloads to `data/food-101/` on first run).

**Expected result:** Train split (75,750 images) and test split (25,250 images) across 101 food classes.

**Validation checks:**
- Train + test split sizes are correct
- Number of unique classes equals 101
- Spot-check: first 10 labels are valid integers in [0, 100]


In [5]:
# ── 2.1  Load Food-101 via torchvision ───────────────────────────────────────
# torchvision.datasets.Food101 downloads to DATA_DIR/food-101/ on first run.
# Subsequent runs load from the local cache — no network needed.
FOOD101_ROOT = DATA_DIR  # downloads into DATA_DIR/food-101/

print("Loading Food-101 (train split) …")
ds_train = tv_datasets.Food101(root=FOOD101_ROOT, split="train", download=True)
print("Loading Food-101 (test split) …")
ds_test  = tv_datasets.Food101(root=FOOD101_ROOT, split="test",  download=True)
print("Done.\n")

# ── Inspect structure ─────────────────────────────────────────────────────────
train_size  = len(ds_train)
test_size   = len(ds_test)
class_names = ds_train.classes          # list of 101 food names
n_classes   = len(class_names)

print(f"{'Split':<15} {'Rows':>8}")
print("-" * 25)
print(f"{'train':<15} {train_size:>8,}")
print(f"{'test':<15} {test_size:>8,}")
print(f"{'total':<15} {train_size + test_size:>8,}")
print()
print(f"Classes : {n_classes}")
print(f"Example labels: {class_names[:5]} … {class_names[-5:]}")

# ── Validation ────────────────────────────────────────────────────────────────
assert train_size == 75_750, f"Expected 75,750 train rows, got {train_size}"
assert test_size  == 25_250, f"Expected 25,250 test rows, got {test_size}"
assert n_classes  == 101,    f"Expected 101 classes, got {n_classes}"

# Spot-check: first 10 labels are valid integers in [0, 100]
sample_labels = [ds_train[i][1] for i in range(10)]
assert all(0 <= l < 101 for l in sample_labels), "Invalid labels found in train split"

print("\n✓ Section 2.1 validation passed — Food-101 loaded correctly.")


Loading Food-101 (train split) …


100%|██████████| 5.00G/5.00G [05:04<00:00, 16.4MB/s]   


Loading Food-101 (test split) …
Done.

Split               Rows
-------------------------
train             75,750
test              25,250
total            101,000

Classes : 101
Example labels: ['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare'] … ['tacos', 'takoyaki', 'tiramisu', 'tuna_tartare', 'waffles']

✓ Section 2.1 validation passed — Food-101 loaded correctly.



### 2.2 — Load WineSensed Reviews

Load the WineSensed dataset from Hugging Face (`Dakhoo/L2T-NeurIPS-2023`, `vintages` config).
No API key required — uses the `datasets` library with `trust_remote_code=True`.

**Dataset:** WineSensed (NeurIPS 2023) — 824k real Vivino user tasting notes tied to 350k+ unique wine vintages,
each annotated with wine name, grape composition, origin, alcohol %, and average Vivino rating.
Licence: CC BY-NC-ND 4.0 (non-commercial research use).

**Key columns:**

| Column | Content |
|---|---|
| `review` | Real human-written Vivino tasting note → renamed `review_text` |
| `wine` | Wine name (e.g. *"Château Margaux"*) |
| `year` | Vintage year → combined with `wine` into `wine_label` |
| `grape` | Comma-separated grape varieties, primary grape first |
| `rating` | Average Vivino rating (0–5) → converted to `rating_pct` (%) |
| `country` / `region` | Wine origin |

**Expected result:** > 100k rows with real review text; `grape` column intact; `rating_pct` computed.

**Validation checks:**
- Row count > 100,000
- `review_text` non-null in > 80% of rows
- `grape` column present
- `rating` values in range [0, 5]


In [ ]:

# ── 2.2  Load WineSensed Reviews ─────────────────────────────────────────────
# Dataset : Dakhoo/L2T-NeurIPS-2023 (Hugging Face) — WineSensed NeurIPS 2023
# Licence : CC BY-NC-ND 4.0 — non-commercial research use
# 824k real Vivino user tasting notes, 350k+ unique wine vintages.
# Columns : review, wine, year, grape, rating, alcohol, country, region, ...

from datasets import load_dataset

print("Loading WineSensed from Hugging Face (vintages config) …")
print("First run downloads ~150 MB — cached on subsequent runs.\n")

# Try 'vintages' first (images_reviews_attributes); fall back to 'all'
for _config in ("vintages", "all"):
    try:
        _ds = load_dataset("Dakhoo/L2T-NeurIPS-2023", _config, trust_remote_code=True)
        print(f"Loaded config : '{_config}'")
        print(f"Splits        : {list(_ds.keys())}")
        break
    except Exception as _e:
        print(f"Config '{_config}' failed: {_e} — trying next …")

# Flatten all splits into one DataFrame
df_wine = pd.concat(
    [_ds[s].to_pandas() for s in _ds.keys()],
    ignore_index=True
)

print(f"\nShape   : {df_wine.shape}")
print(f"Columns : {df_wine.columns.tolist()}")

# ── Normalise column names ────────────────────────────────────────────────────
df_wine.columns = [c.strip().lower().replace(" ", "_") for c in df_wine.columns]

# ── review_text — real Vivino tasting notes ───────────────────────────────────
if "review" in df_wine.columns and "review_text" not in df_wine.columns:
    df_wine = df_wine.rename(columns={"review": "review_text"})

# ── wine_label = wine name + vintage year ─────────────────────────────────────
if "wine" in df_wine.columns and "year" in df_wine.columns:
    df_wine["wine_label"] = (
        df_wine["wine"].fillna("Unknown Wine").astype(str)
        + " "
        + df_wine["year"].fillna("").astype(str).str.strip()
    ).str.strip()
elif "wine" in df_wine.columns:
    df_wine["wine_label"] = df_wine["wine"].fillna("Unknown Wine").astype(str)
else:
    df_wine["wine_label"] = "Unknown Wine"

# ── rating_pct : Vivino avg rating (0–5) → user approval % ──────────────────
if "rating" in df_wine.columns:
    df_wine["rating_pct"] = (
        pd.to_numeric(df_wine["rating"], errors="coerce")
        .clip(0, 5)
        .div(5.0)
        .mul(100)
        .round(0)
        .astype("Int64")     # nullable int — preserves NaN without crashing
    )

# ── Drop rows with no review text ─────────────────────────────────────────────
_before = len(df_wine)
df_wine = df_wine[
    df_wine["review_text"].notna()
    & (df_wine["review_text"].str.strip() != "")
].copy()
print(f"\nDropped {_before - len(df_wine):,} rows with empty review text.")
print(f"Remaining : {len(df_wine):,} rows with real Vivino tasting notes.")

# ── Preview ───────────────────────────────────────────────────────────────────
_PREVIEW = ["wine_label", "review_text", "rating_pct", "grape", "country", "region", "alcohol"]
_preview_cols = [c for c in _PREVIEW if c in df_wine.columns]

pd.set_option("display.max_colwidth", 90)
display(df_wine[_preview_cols].head(5))
pd.reset_option("display.max_colwidth")


Found cached file → data/beerreviews.csv

Shape    : (1586614, 13)

Columns  : ['brewery_id', 'brewery_name', 'review_time', 'review_overall', 'review_aroma', 'review_appearance', 'review_profilename', 'beer_style', 'review_palate', 'review_taste', 'beer_name', 'beer_abv', 'beer_beerid']

Building review_text column (may take ~30 s on 1.5 M rows) …
Done.

Sample review_text:
  Beer group Wheat, sub-group Hefeweizen, brewed by vecchio birraio. This bready yeasty fruity light refreshing spicy clove banana smooth hazy beer is called sausa weizen. Overall average quality with mild aroma, average taste, thin body, and average appearance.
  Beer group unknown, sub-group English Strong Ale, brewed by vecchio birraio. This  beer is called red moon. Overall good quality with mild aroma, good taste, medium body, and decent appearance.


,beer_style,review_text,review_overall,review_aroma,review_taste
0,Hefeweizen,"Beer group Wheat, sub-group Hefeweizen, brewed...",1.5,2.0,1.5
1,English Strong Ale,"Beer group unknown, sub-group English Strong A...",3.0,2.5,3.0
2,Foreign / Export Stout,"Beer group Stout-Porter, sub-group Foreign / E...",3.0,2.5,3.0


In [ ]:

# ── 2.2  Validation ───────────────────────────────────────────────────────────
assert len(df_wine) > 100_000, \
    f"Expected >100k rows with review text, got {len(df_wine):,}"

assert "review_text" in df_wine.columns, "review_text column missing"
assert "grape"       in df_wine.columns, "grape column missing"

null_review = df_wine["review_text"].isnull().sum()
null_pct    = null_review / len(df_wine)
assert null_pct < 0.20, \
    f"Too many null reviews: {null_review:,} ({null_pct:.1%}) — expected < 20%"

if "rating" in df_wine.columns:
    valid_ratings = pd.to_numeric(df_wine["rating"], errors="coerce").dropna()
    assert valid_ratings.between(0, 5).all(), "Ratings outside [0, 5] range"

print("✓ Section 2.2 validation passed")
print(f"  Rows with review text : {len(df_wine):>10,}")
print(f"  Null review_text      : {null_review:>10,}  ({null_pct:.1%})")
if "wine_label" in df_wine.columns:
    print(f"  Unique wines          : {df_wine['wine_label'].nunique():>10,}")
if "grape" in df_wine.columns:
    n_grapes = df_wine["grape"].str.split(",").str[0].str.strip().nunique()
    print(f"  Unique primary grapes : {n_grapes:>10,}")
if "rating_pct" in df_wine.columns:
    rp = df_wine["rating_pct"].dropna()
    print(f"  Rating range          :  {int(rp.min())}% – {int(rp.max())}%  "
          f"(mean {rp.mean():.0f}%)")

avg_words = df_wine["review_text"].str.split().str.len().mean()
print(f"\nreview_text avg length  : {avg_words:.1f} words")
print("\n(Real Vivino tasting notes — no synthetic text.)")


### 2.3 — Select top 15 grape varieties as classification labels

Rather than grouping wines into broad types (Red / White / Rosé), we use the **primary grape variety** directly as the BiLSTM classification target. This forces the model to learn the fine-grained flavor language specific to each grape — *"cassis and cedar"* for Cabernet Sauvignon vs *"strawberry and silk"* for Pinot Noir.

We select the **top 15 grapes by review count** from the `grape` column (first entry in the comma-separated list). These 15 varieties typically cover ~85% of all reviews. Rows whose primary grape is not in the top 15 are dropped.

**Expected result:** New `grape_class` column with one of 15 grape names. Distribution printed.

**Validation checks:**
- Exactly 15 unique values in `grape_class`
- Distribution printed (check for severe imbalance)
- No nulls in `grape_class`


In [ ]:

# ── 2.3  Select top 15 grape varieties as classification labels ───────────────
# Primary grape = first entry in the comma-separated `grape` column.
# We keep the top 15 by review count — covers ~85% of all reviews.
# These become the 15 BiLSTM classes and the 15 Word2Vec grape embedding targets.

TOP_N_GRAPES = 15

# ── Extract primary grape ─────────────────────────────────────────────────────
df_wine["primary_grape"] = (
    df_wine["grape"]
    .fillna("")
    .str.split(",")
    .str[0]
    .str.strip()
)

# ── Find top N by frequency ───────────────────────────────────────────────────
grape_counts  = df_wine["primary_grape"].value_counts()
top_grapes    = grape_counts.head(TOP_N_GRAPES).index.tolist()

print(f"Top {TOP_N_GRAPES} grapes selected (out of "
      f"{df_wine['primary_grape'].nunique()} unique primary grapes):\n")
print(", ".join(top_grapes))

# ── Filter & assign label ─────────────────────────────────────────────────────
df_wine["grape_class"] = df_wine["primary_grape"].where(
    df_wine["primary_grape"].isin(top_grapes)
)
df_wine_mapped = df_wine.dropna(subset=["grape_class"]).copy()

# ── Coverage ──────────────────────────────────────────────────────────────────
coverage = len(df_wine_mapped) / len(df_wine) * 100
print(f"\nRows kept    : {len(df_wine_mapped):>10,}  ({coverage:.1f}% of reviews)")
print(f"Rows dropped : {len(df_wine) - len(df_wine_mapped):>10,}  "
      f"(primary grape not in top {TOP_N_GRAPES})\n")

# ── Validation ────────────────────────────────────────────────────────────────
n_classes = df_wine_mapped["grape_class"].nunique()
assert n_classes == TOP_N_GRAPES, \
    f"Expected {TOP_N_GRAPES} classes, got {n_classes}"
assert df_wine_mapped["grape_class"].isnull().sum() == 0

# ── Distribution ──────────────────────────────────────────────────────────────
dist = df_wine_mapped["grape_class"].value_counts()
print(f"{'Grape variety':<28} {'Count':>8}  {'%':>6}")
print("-" * 46)
for grape, count in dist.items():
    pct = count / len(df_wine_mapped) * 100
    bar = "█" * int(pct / 2)
    print(f"{grape:<28} {count:>8,}  {pct:>5.1f}%  {bar}")

# ── Expose the ordered class list for later sections ─────────────────────────
GRAPE_CLASSES = dist.index.tolist()   # ordered by frequency

print(f"\n✓ Section 2.3 complete — grape_class column added ({TOP_N_GRAPES} classes).")


### 2.4 — Food flavor table

The food flavor table is the bridge between the CNN's food label and the wine pairing system.
It is embedded directly as a Python dictionary — no file dependency.

Each of the 101 Food-101 classes has three sets of flavor keywords:

| Key | Purpose | Example for `pizza` |
|---|---|---|
| `complement` | Flavors that echo and amplify the food's dominant taste — points toward grapes with matching character | `savory tomato earthy herbaceous rich fatty` |
| `contrast` | Flavors that cut through the food and refresh the palate — points toward grapes with high acidity or minerality | `mineral crisp citrus steely dry acidic` |
| `balance` | Approachable, crowd-safe — points toward lighter, fruit-forward grapes | `light fruity clean gentle soft easy` |

Because Word2Vec is **pre-trained on Google News** before being fine-tuned on WineSensed reviews, the flavor table can use natural food language — *tomato*, *fatty*, *smoky*, *creamy* — alongside wine tasting words. All of these have meaningful vectors in the shared space. There is no need to restrict keywords to wine vocabulary only.

At inference time, keywords from the relevant set are embedded using the fine-tuned Word2Vec model (Section 6) and compared by cosine similarity against the 15 grape embedding vectors. The top-matching grape is selected, and the highest-rated real wine of that grape is retrieved with the most representative Vivino review.

**Output for each pairing:**
- Grape variety name (e.g. *Sangiovese*)
- Wine bottle name + vintage (e.g. *Chianti Classico Riserva 2019*)
- Real Vivino tasting note
- User approval % (`rating / 5.0 × 100`)

**Validation checks:**
- All 101 Food-101 class names are present as keys
- Each entry has exactly 3 keys: `complement`, `contrast`, `balance`
- No empty keyword lists
